In [12]:
OPENAI_API_KEY = "sk"

import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [1]:
!pip install textgrad # you might need to restart the notebook after installing textgrad

from textgrad.engine import get_engine
from textgrad import Variable
from textgrad.optimizer import TextualGradientDescent
from textgrad.loss import TextLoss
from dotenv import load_dotenv
load_dotenv()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.2/51.2 kB 993.8 kB/s eta 0:00:00 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.6 MB/s eta 0:00:00
  Created wheel for textgrad: filename=textgrad-0.1.4-py3-none-any.whl size=50063 sha256=2020fd6087920362c1137af604ab2c4a343d3b5ae3887148792a13a6b7b61ced
  Stored in directory: /Users/abishekchiffon/Library/Caches/pip/wheels/d0/e3/65/3fa8dc9fe9b9146f129ebcdcea693c3d5e7cf35a054f74b360
Successfully built textgrad


False

## Introduction: Variable

Variables in TextGrad are the metaphorical equivalent of tensors in PyTorch. They are the primary data structure that you will interact with when using TextGrad.

Variables keep track of gradients and manage the data.

Variables require two arguments (and there is an optional third one):

1. `data`: The data that the variable will hold
2. `role_description`: A description of the role of the variable in the computation graph
3. `requires_grad`: (optional) A boolean flag that indicates whether the variable requires gradients

In [8]:
x = Variable("A sntence with a typo", role_description="The input sentence", requires_grad=True)

In [9]:
x.gradients

set()

## Introduction: Engine

When we talk about the engine in TextGrad, we are referring to an LLM. The engine is an abstraction we use to interact with the model.

In [13]:
engine = get_engine("gpt-3.5-turbo")

This object behaves like you would expect an LLM to behave: You can sample generation from the engine using the `generate` method.

In [14]:
engine.generate("Hello how are you?")

"Hello! I'm here and ready to assist you. How can I help you today?"

## Introduction: Loss

Again, Loss in TextGrad is the metaphorical equivalent of loss in PyTorch. We use Losses in different form in TextGrad but for now we will focus on a simple TextLoss. TextLoss is going to evaluate the loss wrt a string.

In [15]:
system_prompt = Variable("Evaluate the correctness of this sentence", role_description="The system prompt")
loss = TextLoss(system_prompt, engine=engine)

In [16]:
system_prompt

Variable(value=Evaluate the correctness of this sentence, role=The system prompt, grads=)

## Introduction: Optimizer

Keeping on the analogy with PyTorch, the optimizer in TextGrad is the object that will update the parameters of the model. In this case, the parameters are the variables that have `requires_grad` set to `True`.

**NOTE** This is a text optimizer! It will do all operations with text!

In [17]:
optimizer = TextualGradientDescent(parameters=[x], engine=engine)


## Putting it all together

We can now put all the pieces together. We have a variable, an engine, a loss, and an optimizer. We can now run a single optimization step.

In [18]:
l = loss(x)
l.backward(engine)
optimizer.step()

In [20]:
x

Variable(value=A sentence with a typo, role=The input sentence, grads=Here is a conversation:

<CONVERSATION><LM_SYSTEM_PROMPT> Evaluate the correctness of this sentence </LM_SYSTEM_PROMPT>

<LM_INPUT> A sntence with a typo </LM_INPUT>

<LM_OUTPUT> The sentence you provided does indeed contain a typo. The correct spelling should be "sentence" instead of "sntence." </LM_OUTPUT>

</CONVERSATION>

This conversation is potentially part of a larger system. The output is used as response from the language model

Here is the feedback we got for The input sentence in the conversation:

<FEEDBACK>Since the language model correctly identified a typo in the sentence provided, the feedback for the variable "<VARIABLE> A sntence with a typo </VARIABLE>" would be to pay closer attention to spelling errors. To improve the <OBJECTIVE_FUNCTION>, it is crucial to ensure that the text is free of typos by carefully proofreading before finalizing the input sentence. One strategy to enhance the quality of t

In [19]:
x.value

'A sentence with a typo'

In [21]:
optimizer.

While here it is not going to be useful, we can also do multiple optimization steps in a loop! Do not forget to reset the gradients after each step!

In [ ]:
optimizer.zero_grad()